In [ ]:
%load_ext autoreload
%autoreload 2

# Pefrom Wiener-SVD Unfolding

In [ ]:
from os import path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')
from analysis_village.unfolding.utils import *

from var_config_cohpi import *


## 1. Open Covariance Files

In [ ]:
genie_cov = np.load("genie_syst_cov_matrices.npz", allow_pickle=True)

In [ ]:
genie_cov['cov_ms_ms'].item()['cov']

## 2. Asimov data test

### 2 - 1: Make Asimov data

In [ ]:
## for asimov data stat unc. calculation
XSEC_UNIT = 7.038e-43

asimov_data_sr = genie_cov['bs'] + genie_cov['ms']
asimov_data_sb = genie_cov['bc'] + genie_cov['mc']
## stat. unc of asimov data in sqrt(N) approximation, converted to xsec unit
cov_asimov_data_sr = np.diag(asimov_data_sr * XSEC_UNIT)
cov_asimov_data_sb = np.diag(asimov_data_sb * XSEC_UNIT)

### 2 - 2: Plot Asimov data vs. MC plot

In [ ]:
plot_overlay_with_cov(
    data=asimov_data_sr,
    cov_data=cov_asimov_data_sr,
    signal=genie_cov["ms"],
    cov_signal=genie_cov["cov_ms_ms"].item()["cov"],
    background=genie_cov["bs"],
    cov_background=genie_cov["cov_bs_bs"].item()["cov"],
    varcfg=varcfg_p_mu,
    title="SR: Asimov vs Prediction (GENIE syst)",
)


In [ ]:
plot_overlay_with_cov(
    data=asimov_data_sb,
    cov_data=cov_asimov_data_sb,
    signal=genie_cov["mc"],
    cov_signal=genie_cov["cov_mc_mc"].item()["cov"],
    background=genie_cov["bc"],
    cov_background=genie_cov["cov_bc_bc"].item()["cov"],
    varcfg=varcfg_p_mu,
    title="SB: Asimov vs Prediction (GENIE syst)",
)


### 2 - 3: CCBC

Merge SB asimov data's stat unc to total cov in SB

In [ ]:
cov_cc = genie_cov["cov_mc_mc"].item()["cov"] + genie_cov["cov_bc_bc"].item()["cov"] + genie_cov["cov_mc_bc"].item()["cov"] + genie_cov["cov_bc_mc"].item()["cov"]
tilde_cov_cc = cov_cc + cov_asimov_data_sb

Constrained bs and cov_bs_bs

In [ ]:
## since d_C - n_c^CV is zero, bs_const = genie_cov["bs"]
cov_bs_nc = (genie_cov["cov_bs_bc"].item()["cov"] + genie_cov["cov_bs_mc"].item()["cov"])
cov_bs_nc_T = cov_bs_nc.T
tilde_cov_cc_inv = collect_inv_cov(tilde_cov_cc)
bs_const = genie_cov["bs"] + cov_bs_nc @ tilde_cov_cc_inv @ (asimov_data_sb - genie_cov['bc'] - genie_cov['mc'])

In [ ]:
cov_bs_bs_const = genie_cov["cov_bs_bs"].item()["cov"] - cov_bs_nc @ tilde_cov_cc_inv @ cov_bs_nc_T

Unconst vs. Const for Asimov - background: merge asimov stat unc to bkg cov

In [ ]:
n_bins = len(varcfg_p_mu.bins) - 1
cov_zero = np.zeros((n_bins, n_bins))
arr_zero = np.zeros(n_bins)

asimov_m_bkg_sr = asimov_data_sr - genie_cov["bs"]
asimov_m_bkg_sr_const = asimov_data_sr - bs_const

cov_asimov_m_bkg_sr = genie_cov["cov_bs_bs"].item()["cov"] + cov_asimov_data_sr
cov_asimov_m_bkg_sr_const = cov_bs_bs_const + cov_asimov_data_sr

In [ ]:
plot_overlay_with_cov(
    data=asimov_m_bkg_sr,
    cov_data=cov_asimov_m_bkg_sr,
    signal=arr_zero,
    cov_signal=cov_zero,
    background=asimov_m_bkg_sr_const,
    cov_background= cov_asimov_m_bkg_sr_const,
    varcfg=varcfg_p_mu,
    title="SR: Asimov - Background (GENIE syst)",
    ylims=(-1e-41, 6e-41),
    bkg_label="Asimov - bkg",
    sig_label="Nothing",
    sig_p_bkg_label="",
    pred_unc_label="Post-CCBC Unc.",
    data_label="Pre-CCBC Unc."
)
